# Notebook 01 — Data Extraction & Initial Profiling

**Project:** Used Vehicle Pricing Intelligence
**Dataset:** Manheim Used Vehicle Sales Dataset
**Source:** Kaggle — raw transaction-level used car sales records
**Objective:** Load raw dataset, assess structure and quality, and document initial data profile for Gate 1 review.

In [36]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 120)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Dataset Source & Rationale

The dataset contains **558,837 transaction-level records** of used vehicle sales from the Manheim auction platform. It covers sales across multiple US states and includes vehicle attributes, seller information, and market pricing benchmarks (MMR — Manheim Market Report). The dataset was sourced from Kaggle in its raw, unprocessed form and contains realistic data quality issues suitable for ETL pipeline development.

**Why this dataset?**
- Row count (558,837) far exceeds the 5,000-row minimum
- 16 meaningful columns spanning categorical and numerical data types
- Contains missing values, inconsistent formatting, and outlier values
- Directly relevant to the business problem of used vehicle pricing intelligence

In [37]:
# Load raw dataset
RAW_PATH = '../data/raw/car_prices.csv'

df = pd.read_csv(RAW_PATH)

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nColumn Data Types:")
print(df.dtypes)

Dataset loaded successfully.
Shape: 558,837 rows x 16 columns

Column Data Types:
year              int64
make                str
model               str
trim                str
body                str
transmission        str
vin                 str
state               str
condition       float64
odometer        float64
color               str
interior            str
seller              str
mmr             float64
sellingprice    float64
saledate            str
dtype: object


## 2. Initial Data Inventory

In [38]:
print("=" * 60)
print("DATASET INVENTORY")
print("=" * 60)

print(f"\nShape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\nData Types:")
print(df.dtypes.to_string())

print(f"\nMemory Usage:")
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"  Total: {mem_mb:.2f} MB")

print("\nFirst 5 rows:")
df.head()

DATASET INVENTORY

Shape: (558837, 16)

Columns (16):
   1. year
   2. make
   3. model
   4. trim
   5. body
   6. transmission
   7. vin
   8. state
   9. condition
  10. odometer
  11. color
  12. interior
  13. seller
  14. mmr
  15. sellingprice
  16. saledate

Data Types:
year              int64
make                str
model               str
trim                str
body                str
transmission        str
vin                 str
state               str
condition       float64
odometer        float64
color               str
interior            str
seller              str
mmr             float64
sellingprice    float64
saledate            str

Memory Usage:
  Total: 371.54 MB

First 5 rows:


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.00,"16,639.00",white,black,kia motors america inc,"20,500.00","21,500.00",Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.00,"9,393.00",white,beige,kia motors america inc,"20,800.00","21,500.00",Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.00,"1,331.00",gray,black,financial services remarketing (lease),"31,900.00","30,000.00",Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.00,"14,282.00",white,black,volvo na rep/world omni,"27,500.00","27,750.00",Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.00,"2,641.00",gray,black,financial services remarketing (lease),"66,000.00","67,000.00",Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


## 3. Missing Value Audit

In [39]:
print("=" * 60)
print("MISSING VALUE AUDIT")
print("=" * 60)

missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_count.index,
    'Missing Count': missing_count.values,
    'Missing %': missing_pct.values
}).sort_values('Missing Count', ascending=False)

missing_df = missing_df[missing_df['Missing Count'] > 0].reset_index(drop=True)

print(f"\nColumns with missing values: {len(missing_df)} / {len(df.columns)}")
print("\nMissing Value Summary:")
print(missing_df.to_string(index=False))

# Known reference values for validation
print("\n--- Expected Reference Values ---")
expected = {
    'make': (10301, 1.84),
    'model': (10399, 1.86),
    'trim': (10651, 1.91),
    'body': (13195, 2.36),
    'transmission': (65352, 11.69),
    'condition': (11820, 2.12),
    'odometer': (94, 0.02),
    'color': (749, 0.13),
    'interior': (749, 0.13),
    'mmr': (38, 0.01),
    'sellingprice': (12, 0.00)
}
for col, (cnt, pct) in expected.items():
    print(f"  {col}: {cnt:,} ({pct}%)")

MISSING VALUE AUDIT

Columns with missing values: 13 / 16

Missing Value Summary:
      Column  Missing Count  Missing %
transmission          65352      11.69
        body          13195       2.36
   condition          11820       2.12
        trim          10651       1.91
       model          10399       1.86
        make          10301       1.84
       color            749       0.13
    interior            749       0.13
    odometer             94       0.02
         mmr             38       0.01
sellingprice             12       0.00
    saledate             12       0.00
         vin              4       0.00

--- Expected Reference Values ---
  make: 10,301 (1.84%)
  model: 10,399 (1.86%)
  trim: 10,651 (1.91%)
  body: 13,195 (2.36%)
  transmission: 65,352 (11.69%)
  condition: 11,820 (2.12%)
  odometer: 94 (0.02%)
  color: 749 (0.13%)
  interior: 749 (0.13%)
  mmr: 38 (0.01%)
  sellingprice: 12 (0.0%)


## 4. Statistical Summary (Numerical Columns)

In [40]:
print("=" * 60)
print("NUMERICAL COLUMN SUMMARY STATISTICS")
print("=" * 60)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumerical columns ({len(num_cols)}): {num_cols}")

print("\nDescriptive Statistics:")
print(df[num_cols].describe().T.to_string())

print("\nKey Observations:")
if 'sellingprice' in df.columns:
    print(f"  sellingprice range: ${df['sellingprice'].min():,.0f} — ${df['sellingprice'].max():,.0f}")
    print(f"  sellingprice mean:  ${df['sellingprice'].mean():,.2f}")
if 'mmr' in df.columns:
    print(f"  mmr mean:           ${df['mmr'].mean():,.2f}")
if 'odometer' in df.columns:
    print(f"  odometer max:       {df['odometer'].max():,.0f} miles")
if 'year' in df.columns:
    print(f"  year range:         {df['year'].min()} — {df['year'].max()}")

NUMERICAL COLUMN SUMMARY STATISTICS

Numerical columns (5): ['year', 'condition', 'odometer', 'mmr', 'sellingprice']

Descriptive Statistics:
                  count      mean       std      min       25%       50%       75%        max
year         558,837.00  2,010.04      3.97 1,982.00  2,007.00  2,012.00  2,013.00   2,015.00
condition    547,017.00     30.67     13.40     1.00     23.00     35.00     42.00      49.00
odometer     558,743.00 68,320.02 53,398.54     1.00 28,371.00 52,254.00 99,109.00 999,999.00
mmr          558,799.00 13,769.38  9,679.97    25.00  7,100.00 12,250.00 18,300.00 182,000.00
sellingprice 558,825.00 13,611.36  9,749.50     1.00  6,900.00 12,100.00 18,200.00 230,000.00

Key Observations:
  sellingprice range: $1 — $230,000
  sellingprice mean:  $13,611.36
  mmr mean:           $13,769.38
  odometer max:       999,999 miles
  year range:         1982 — 2015


## 5. Categorical Column Distributions

In [41]:
cat_cols = ['make', 'body', 'transmission', 'state', 'color']

for col in cat_cols:
    if col in df.columns:
        print(f"\n{'=' * 50}")
        print(f"Top 10 values — {col.upper()}")
        print(f"{'=' * 50}")
        vc = df[col].value_counts().head(10)
        for val, cnt in vc.items():
            pct = cnt / len(df) * 100
            print(f"  {str(val):<25} {cnt:>8,}  ({pct:.1f}%)")


Top 10 values — MAKE
  Ford                        93,554  (16.7%)
  Chevrolet                   60,197  (10.8%)
  Nissan                      53,946  (9.7%)
  Toyota                      39,871  (7.1%)
  Dodge                       30,710  (5.5%)
  Honda                       27,206  (4.9%)
  Hyundai                     21,816  (3.9%)
  BMW                         20,719  (3.7%)
  Kia                         18,077  (3.2%)
  Chrysler                    17,276  (3.1%)

Top 10 values — BODY
  Sedan                      199,437  (35.7%)
  SUV                        119,292  (21.3%)
  sedan                       41,906  (7.5%)
  suv                         24,552  (4.4%)
  Hatchback                   21,380  (3.8%)
  Minivan                     21,363  (3.8%)
  Coupe                       14,602  (2.6%)
  Wagon                       13,630  (2.4%)
  Crew Cab                    13,280  (2.4%)
  Convertible                  8,652  (1.5%)

Top 10 values — TRANSMISSION
  automatic           

## 6. Data Quality Observations

| Issue | Column(s) | Action Required |
|-------|-----------|----------------|
| Mixed case categories | make, body, transmission | Standardise to Title Case / lowercase |
| High missingness | transmission (11.69%) | Impute from make/body mode |
| Outlier prices | sellingprice, mmr | IQR-based winsorisation |
| Invalid values | odometer (999,999) | Cap at realistic maximum |
| Date parsing | saledate | Convert to datetime |
| Year range | year (min=1982) | Filter pre-1990 vehicles |

**Dataset Verdict:** Approved for ETL pipeline. Realistic cleaning challenges present across multiple columns. Proceed to Notebook 02.

In [42]:
# Save data profile summary
profile_summary = {
    'raw_rows': int(df.shape[0]),
    'raw_cols': int(df.shape[1]),
    'columns': df.columns.tolist(),
    'dtypes': df.dtypes.astype(str).to_dict(),
    'missing_counts': df.isnull().sum().to_dict(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2).to_dict(),
    'num_summary': {
        col: {
            'mean': float(df[col].mean()) if df[col].dtype in [np.float64, np.int64] else None,
            'min': float(df[col].min()) if df[col].dtype in [np.float64, np.int64] else None,
            'max': float(df[col].max()) if df[col].dtype in [np.float64, np.int64] else None,
        }
        for col in df.select_dtypes(include=[np.number]).columns
    }
}

print("Data profile summary created.")
print(f"  Total rows: {profile_summary['raw_rows']:,}")
print(f"  Total columns: {profile_summary['raw_cols']}")
print(f"  Columns with missing data: {sum(1 for v in profile_summary['missing_counts'].values() if v > 0)}")
print()
print("✅ Extraction complete. Raw dataset ready for ETL pipeline (Notebook 02).")

Data profile summary created.
  Total rows: 558,837
  Total columns: 16
  Columns with missing data: 13

✅ Extraction complete. Raw dataset ready for ETL pipeline (Notebook 02).
